In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
in_dir = '../../results/03_accessbpnet/08_run_tfmodisco'
out_dir = '../../results/03_accessbpnet/10_extract_tfmodisco'

os.makedirs(out_dir, exist_ok=True)

In [ ]:
df_list = []
for cell_type in ['K562', 'HepG2']:
    for assay in ['ATAC', 'ACCESS']:
        for output in ['counts', 'profile']:
            score_file = f'{in_dir}/{cell_type}/{assay}/{output}_scores.h5'

            scores = h5py.File(score_file, 'r')
            names, n_seqlets_list = [], []
            for name in scores['pos_patterns'].keys():
                n_seqlets = np.array(scores['pos_patterns'][name]['seqlets']['n_seqlets'])[0]
                name = name.split("_")[-1]
                names.append(name)
                n_seqlets_list.append(n_seqlets)

            df = pd.DataFrame(data={'pos_patterns': names, 
                                    'n_seqlets': n_seqlets_list})
            df['pos_patterns'] = df['pos_patterns'].astype(int)
            df = df.sort_values(['pos_patterns'])
            df['cell_type'] = cell_type
            df['assay'] = assay
            df['output'] = output

            df_list.append(df)

In [ ]:
df = pd.concat(df_list)

In [ ]:
df = df[df['pos_patterns'] < 10]

In [ ]:
df.to_csv(f'{out_dir}/tfmodisco_seqlet_counts_per_pattern.csv', index=False)

In [ ]:
df = df[df['output'] == 'counts']

In [ ]:
df1 = df[df['cell_type'] == 'K562']
df2 = df[df['cell_type'] == 'HepG2']

In [ ]:
sns.barplot(data=df1, x='pos_patterns', y='n_seqlets', hue='assay')

In [ ]:
sns.barplot(data=df2, x='pos_patterns', y='n_seqlets', hue='assay')

In [ ]:
# plot motif logos for top patterns
trim_threshold = 0.3
for cell_type in ['K562']:
    for assay in ['ATAC']:
        for output in ['counts']:
            score_file = f'{in_dir}/{cell_type}/{assay}/{output}_scores.h5'

            scores = h5py.File(score_file, 'r')
            names, n_seqlets_list = [], []
            for name in scores['pos_patterns'].keys():
                cwm_fwd = np.array(scores['pos_patterns'][name]['contrib_scores'][:])
                cwm_rev = cwm_fwd[::-1, ::-1]

                score_fwd = np.sum(np.abs(cwm_fwd), axis=1)
                score_rev = np.sum(np.abs(cwm_rev), axis=1)
                
                trim_thresh_fwd = np.max(score_fwd) * trim_threshold
                trim_thresh_rev = np.max(score_rev) * trim_threshold

                pass_inds_fwd = np.where(score_fwd >= trim_thresh_fwd)[0]
                pass_inds_rev = np.where(score_rev >= trim_thresh_rev)[0]

                start_fwd, end_fwd = max(np.min(pass_inds_fwd) - 4, 0), min(np.max(pass_inds_fwd) + 4 + 1, len(score_fwd) + 1)
                start_rev, end_rev = max(np.min(pass_inds_rev) - 4, 0), min(np.max(pass_inds_rev) + 4 + 1, len(score_rev) + 1)

                trimmed_cwm_fwd = cwm_fwd[start_fwd:end_fwd]
                trimmed_cwm_rev = cwm_rev[start_rev:end_rev]

                _plot_weights(trimmed_cwm_fwd, path=f'{out_dir}/{name}.cwm.fwd.pdf')